# RCAEval 无链路示例（仅 Metrics）

这个示例不依赖 traces/logs，只使用 metrics 时序数据完成根因定位。

适用场景：你只有 data.csv（含 time 列和指标列），没有链路数据。

In [2]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd

from RCAEval.utility import download_data, read_data
from RCAEval.e2e import baro, nsigma

## 1. 准备数据

若当前目录没有 data.csv，则自动下载示例数据。

In [3]:
if not os.path.exists('data.csv'):
    download_data()

data = read_data('data.csv')

print('shape =', data.shape)
print('time column exists =', 'time' in data.columns)
data.head()

shape = (721, 48)
time column exists = True


,time,adservice_cpu,cartservice_cpu,checkoutservice_cpu,currencyservice_cpu,emailservice_cpu,frontend_cpu,main_cpu,paymentservice_cpu,productcatalogservice_cpu,...,adservice_latency,cartservice_latency,checkoutservice_latency,currencyservice_latency,emailservice_latency,frontend_latency,paymentservice_latency,productcatalogservice_latency,recommendationservice_latency,shippingservice_latency
0,1692568979,0.948645,5.446051,1.013187,8.401782,0.724534,15.563824,7.865554,0.317490,6.008795,...,0.004301,0.006528,0.047614,0.004510,0.0046,0.099452,0.004373,0.003914,0.008305,0.004243
1,1692568980,0.948645,5.446051,0.944733,7.190852,0.699150,15.563824,7.865554,0.317490,6.870768,...,0.004301,0.006504,0.047598,0.004510,0.0046,0.099452,0.004381,0.003927,0.008192,0.004247
2,1692568981,0.948645,5.446051,0.944733,7.050822,0.699150,15.563824,7.865554,0.317490,6.735170,...,0.004301,0.006504,0.047598,0.004510,0.0046,0.105744,0.004381,0.003927,0.008192,0.004247
3,1692568982,0.948645,5.446051,0.944733,8.042358,0.699150,15.563824,7.865554,0.317490,6.599572,...,0.004301,0.006504,0.047598,0.004510,0.0046,0.099474,0.004396,0.003927,0.008192,0.004247
4,1692568983,0.948645,5.446051,0.944733,8.042358,0.699150,15.461252,7.865554,0.321451,6.463974,...,0.004306,0.006746,0.047632,0.004536,0.0046,0.104210,0.004380,0.003923,0.008251,0.004233


## 2. 设置异常起点并执行 RCA

这里用时间序列中点作为 inject_time 示例。
实际使用时建议替换为真实告警时间。

In [4]:
inject_time = int(data['time'].iloc[len(data) // 2])
print('inject_time =', inject_time)

baro_out = baro(data, inject_time=inject_time)
baro_top10 = baro_out['ranks'][:10]

print('BARO Top 10:')
for i, item in enumerate(baro_top10, 1):
    print(f'{i:02d}. {item}')

inject_time = 1692569339
BARO Top 10:
01. emailservice_mem
02. recommendationservice_mem
03. cartservice_mem
04. checkoutservice_latency
05. cartservice_latency
06. cartservice_cpu
07. recommendationservice_workload
08. cartservice_workload
09. adservice_workload
10. frontend-external_workload


In [5]:
nsigma_out = nsigma(data, inject_time=inject_time)
nsigma_top10 = nsigma_out['ranks'][:10]

print('NSigma Top 10:')
for i, item in enumerate(nsigma_top10, 1):
    print(f'{i:02d}. {item}')

NSigma Top 10:
01. emailservice_mem
02. cartservice_mem
03. cartservice_cpu
04. cartservice_latency
05. checkoutservice_latency
06. recommendationservice_mem
07. recommendationservice_workload
08. cartservice_workload
09. time
10. frontend_workload


In [7]:
compare_df = pd.DataFrame({
    'BARO': baro_top10,
    'NSigma': nsigma_top10
})
compare_df

,BARO,NSigma
0,emailservice_mem,emailservice_mem
1,recommendationservice_mem,cartservice_mem
2,cartservice_mem,cartservice_cpu
3,checkoutservice_latency,cartservice_latency
4,cartservice_latency,checkoutservice_latency
5,cartservice_cpu,recommendationservice_mem
6,recommendationservice_workload,recommendationservice_workload
7,cartservice_workload,cartservice_workload
8,adservice_workload,time
9,frontend-external_workload,frontend_workload


## 2.1 融合 CMDB 做结果重排（可选）

说明：当前这个无链路示例里，BARO/NSigma 主要基于指标异常分数。
如果你有 CMDB（服务重要性、业务优先级、owner 等），推荐在排序结果上做一层加权重排。

In [8]:
# 方式 A：直接在代码里给 CMDB 权重（0~1，越大表示业务越关键）
cmdb_weights = {
    # "checkoutservice": 1.0,
    # "paymentservice": 0.9,
    # "frontend": 0.8,
}

# 方式 B：从文件读取（任选其一）
# 1) JSON 示例: {"checkoutservice": 1.0, "paymentservice": 0.9}
# import json
# with open("cmdb_weights.json", "r", encoding="utf-8") as f:
#     cmdb_weights = json.load(f)

# 2) CSV 示例: 两列 service, weight
# cmdb_df = pd.read_csv("cmdb_weights.csv")
# cmdb_weights = dict(zip(cmdb_df["service"], cmdb_df["weight"]))


def metric_to_service(metric_name: str) -> str:
    if "_" in metric_name:
        return metric_name.split("_", 1)[0]
    return metric_name


def rerank_with_cmdb(ranks, cmdb_map, alpha=0.8):
    """
    alpha: 算法分数权重（0~1）
    1-alpha: CMDB 权重
    """
    n = len(ranks)
    items = []
    for idx, metric in enumerate(ranks):
        service = metric_to_service(metric)

        # 用名次近似算法原始分数：越靠前越高
        algo_score = (n - idx) / n

        # CMDB 权重缺失时默认 0
        cmdb_score = float(cmdb_map.get(service, 0.0))

        final_score = alpha * algo_score + (1 - alpha) * cmdb_score
        items.append((metric, service, algo_score, cmdb_score, final_score))

    out = pd.DataFrame(
        items,
        columns=["metric", "service", "algo_score", "cmdb_score", "final_score"],
    )
    return out.sort_values("final_score", ascending=False).reset_index(drop=True)


# 用 BARO 全量结果做 CMDB 重排
cmdb_rank_df = rerank_with_cmdb(baro_out["ranks"], cmdb_weights, alpha=0.8)
cmdb_rank_df.head(10)

,metric,service,algo_score,cmdb_score,final_score
0,emailservice_mem,emailservice,1.000000,0.0,0.800000
1,recommendationservice_mem,recommendationservice,0.979167,0.0,0.783333
2,cartservice_mem,cartservice,0.958333,0.0,0.766667
3,checkoutservice_latency,checkoutservice,0.937500,0.0,0.750000
4,cartservice_latency,cartservice,0.916667,0.0,0.733333
5,cartservice_cpu,cartservice,0.895833,0.0,0.716667
6,recommendationservice_workload,recommendationservice,0.875000,0.0,0.700000
7,cartservice_workload,cartservice,0.854167,0.0,0.683333
8,adservice_workload,adservice,0.833333,0.0,0.666667
9,frontend-external_workload,frontend-external,0.812500,0.0,0.650000


使用建议：

1. 若你更相信算法排序，把 alpha 调大（例如 0.85~0.95）。
2. 若你更希望优先关键业务服务，把 alpha 调小（例如 0.6~0.75）。
3. 推荐先看 cmdb_rank_df 的 Top10，再和原始 BARO Top10 做交集分析。

In [9]:
# 可选：如果你的 CMDB 里还有 owner 信息，可以做告警归属聚合
# CSV 示例列: service, owner, weight
# owner_df = pd.read_csv("cmdb_service_owner.csv")
# owner_map = dict(zip(owner_df["service"], owner_df["owner"]))

# 下面给一个示例映射，你可替换成真实 CMDB
owner_map = {
    # "checkoutservice": "team-checkout",
    # "paymentservice": "team-payment",
    # "frontend": "team-frontend",
}

if owner_map:
    owner_rank = (
        cmdb_rank_df.assign(owner=lambda x: x["service"].map(owner_map).fillna("unknown"))
        .groupby("owner", as_index=False)["final_score"]
        .sum()
        .sort_values("final_score", ascending=False)
    )
    owner_rank
else:
    print("owner_map 为空：如需 owner 归属聚合，请填充 CMDB owner 映射。")

owner_map 为空：如需 owner 归属聚合，请填充 CMDB owner 映射。


## 3. 下一步建议

1. 将 inject_time 改成真实故障开始时间。
2. 用你自己的 metrics 文件替换 data.csv。
3. 先看 BARO/NSigma 交集，再做人工排查。